# CLUES: Question Disambiguation for Epidemiological Research

**Authors**: Angelo Ziletti, Leonardo D'Ambrosi

**Paper**: *Disentangling Ambiguity from Instability in Large Language Models: A Clinical Text-to-SQL Case Study*  
**arXiv**: [https://arxiv.org/abs/2602.12015](https://arxiv.org/abs/2602.12015)  
Accepted at the [8th Clinical NLP Workshop (ClinicalNLP 2026)](https://clinical-nlp.github.io/2026/) at LREC 2026.

---

Epidemiological questions in natural language frequently contain **implicit ambiguities** that affect query results (see Paper, Section 4.1 and Table 2). For example:

- Does *"patients over 18"* refer to age **at diagnosis** or **current age**?
- Does *"never took heparin before"* mean strictly **before** the diagnosis date, or **on or before**?
- Should we count **first diagnosis** or **any diagnosis**?

CLUES provides two complementary disambiguation functions:

1. **`generate_question_interpretations`** — Guided disambiguation that identifies specific ambiguity types, assigns an ambiguity score, and generates clinically plausible interpretations ordered from most to least likely.

2. **`generate_explore_meanings`** — Exploratory interpretation generation that maximizes semantic diversity for uncertainty estimation (used internally for H(I) computation).

## Setup

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import asyncio
import nest_asyncio
nest_asyncio.apply()

import logging
logging.basicConfig(level=logging.WARNING)

from clues import GeminiClient, generate_question_interpretations, generate_explore_meanings

client = GeminiClient()
print("CLUES loaded successfully.")

CLUES loaded successfully.


## Example Questions

We use four epidemiological questions with varying levels of ambiguity. The first two are from Table 2 of the paper.

In [2]:
QUESTIONS = [
    "How many patients > 17 yo have atopic dermatitis (all codes). Breakdown by code",
    "How many patients with chronic kidney disease never took heparin before their chronic kidney disease diagnosis?",
    "How many patients over 18 have atopic dermatitis?",
    "Among people with CKD, which percentage will develop ischemic stroke within 5 years of diagnosis? Group by year after diagnosis",
]

for i, q in enumerate(QUESTIONS, 1):
    print(f"  {i}. {q}")

  1. How many patients > 17 yo have atopic dermatitis (all codes). Breakdown by code
  2. How many patients with chronic kidney disease never took heparin before their chronic kidney disease diagnosis?
  3. How many patients over 18 have atopic dermatitis?
  4. Among people with CKD, which percentage will develop ischemic stroke within 5 years of diagnosis? Group by year after diagnosis


## 1. Guided Disambiguation

`generate_question_interpretations` analyzes the question for common epidemiological ambiguities:
- Patient count semantics (unique patients vs. all records)
- Temporal relationships (before/after, within timeframes)
- Population definitions (inclusion/exclusion criteria)
- Event ordering (first vs. any occurrence)
- Demographic specifications (age at diagnosis vs. current age)

It returns an **ambiguity score** (0–1) and a list of **interpretations** ordered from most to least clinically plausible.

In [3]:
for i, question in enumerate(QUESTIONS, 1):
    print(f"{'='*70}")
    print(f"Example {i}: {question}")
    print(f"{'='*70}")

    result = asyncio.get_event_loop().run_until_complete(
        generate_question_interpretations(question, client=client)
    )

    print(f"  Ambiguous: {result.is_ambiguous}")
    print(f"  Ambiguity score: {result.ambiguity_score:.2f}")
    print(f"  Interpretations ({len(result.final_interpretations)}):")
    for j, interp in enumerate(result.final_interpretations, 1):
        print(f"    {j}. {interp}")
    print()

Example 1: How many patients > 17 yo have atopic dermatitis (all codes). Breakdown by code


  Ambiguous: True
  Ambiguity score: 0.40
  Interpretations (3):
    1. Provide the count of unique patients for each specific medical code categorized under atopic dermatitis, including only patients who were older than 17 years of age at the time the diagnosis was recorded.
    2. Provide the count of unique patients for each specific medical code categorized under atopic dermatitis, including only patients whose current age is older than 17 years, regardless of their age when the diagnosis occurred.
    3. Provide the total count of unique patients with any atopic dermatitis code who are currently over 17, then list the patient count for every specific code found in their records (where one patient may be counted under multiple codes).

Example 2: How many patients with chronic kidney disease never took heparin before their chronic kidney disease diagnosis?


  Ambiguous: True
  Ambiguity score: 0.30
  Interpretations (2):
    1. How many unique patients with a chronic kidney disease diagnosis have no records of heparin administration dated earlier than their earliest chronic kidney disease diagnosis date?
    2. How many unique patients with a chronic kidney disease diagnosis have no records of heparin administration at any point in their entire medical history?

Example 3: How many patients over 18 have atopic dermatitis?


  Ambiguous: True
  Ambiguity score: 0.40
  Interpretations (2):
    1. How many unique patients are currently older than 18 years of age and have at least one recorded diagnosis of atopic dermatitis in their medical history?
    2. How many unique patients were older than 18 years of age at the time of their first recorded diagnosis of atopic dermatitis?

Example 4: Among people with CKD, which percentage will develop ischemic stroke within 5 years of diagnosis? Group by year after diagnosis


  Ambiguous: True
  Ambiguity score: 0.50
  Interpretations (4):
    1. Among unique patients newly diagnosed with CKD, what percentage experienced their first ischemic stroke in each specific year (Year 1, Year 2, Year 3, Year 4, and Year 5) following their diagnosis date, using the total CKD cohort as the constant denominator?
    2. Among unique patients newly diagnosed with CKD, what is the cumulative percentage of patients who have developed at least one ischemic stroke by the end of each year (Year 1 through Year 5) following their diagnosis?
    3. For each year (1 through 5) following a first CKD diagnosis, what percentage of patients who have not yet had a stroke (the population at risk) develop an ischemic stroke during that specific year?
    4. Among patients with any diagnosis of CKD, what is the total count of ischemic stroke events divided by the number of patients, grouped by the integer number of years (0-5) elapsed since the patient's first CKD diagnosis?



## 2. Exploratory Meanings

`generate_explore_meanings` generates diverse semantic interpretations **without** domain constraints, maximizing breadth. This is used internally by CLUES for H(I) computation (interpretation entropy), but can also be useful on its own to surface unexpected readings of a question.

In [4]:
for i, question in enumerate(QUESTIONS, 1):
    print(f"{'='*70}")
    print(f"Example {i}: {question}")
    print(f"{'='*70}")

    explore = asyncio.get_event_loop().run_until_complete(
        generate_explore_meanings(question, num_interpretations=5, client=client)
    )

    print(f"  Interpretations ({len(explore.interpretations)}):")
    for j, interp in enumerate(explore.interpretations, 1):
        print(f"    {j}. {interp}")
    print()

Example 1: How many patients > 17 yo have atopic dermatitis (all codes). Breakdown by code


  Interpretations (5):
    1. Count all unique patients aged 18 and older who have at least one documented instance of atopic dermatitis or its subtypes in their entire medical history, grouped by the specific clinical label used.
    2. Identify patients currently aged 18 or older who have received a diagnosis of atopic dermatitis within the most recent calendar year, providing a total count for each distinct diagnostic description.
    3. Calculate the total number of patients who were 18 years of age or older at the specific time they were first diagnosed with atopic dermatitis, broken down by the specific type of dermatitis recorded.
    4. Count the number of unique patients aged 18 and older who have had multiple clinical encounters associated with atopic dermatitis, categorized by the specific diagnosis terms found in their records.
    5. Determine the number of patients currently over the age of 17 who have atopic dermatitis listed as an active condition in their problem list,

  Interpretations (5):
    1. Count of patients with any stage of chronic kidney disease who lack any documentation of heparin administration in their medical record prior to the date of their first chronic kidney disease diagnosis.
    2. Number of patients diagnosed with chronic kidney disease who have no recorded orders for unfractionated heparin or low-molecular-weight heparin at any point in their clinical history before the initial chronic kidney disease diagnosis event.
    3. Count of patients with a chronic kidney disease diagnosis who did not have a heparin-related medication entry in the pharmacy or administration records for at least three years preceding their earliest chronic kidney disease record.
    4. Number of patients with chronic kidney disease whose longitudinal record contains zero instances of heparin use occurring earlier than the first time a chronic kidney disease condition was noted in their medical file.
    5. Count of unique patients with a documented dia

  Interpretations (5):
    1. Count of all unique patients aged 18 years or older who have at least one recorded instance of atopic dermatitis in their medical history.
    2. Count of patients who were over the age of 18 at the time of their most recent clinical visit where atopic dermatitis was documented.
    3. Count of adult patients aged 18 and above who have received a new diagnosis of atopic dermatitis within the past twelve months.
    4. Count of unique patients aged 18 or older who have two or more separate clinical entries for atopic dermatitis to confirm a chronic condition.
    5. Count of patients currently aged 18 or older who have an active, unresolved diagnosis of atopic dermatitis in their current problem list.

Example 4: Among people with CKD, which percentage will develop ischemic stroke within 5 years of diagnosis? Group by year after diagnosis


  Interpretations (5):
    1. Calculate the cumulative percentage of patients with a chronic kidney disease diagnosis who experience an ischemic stroke at the one, two, three, four, and five-year marks following their initial diagnosis.
    2. For each individual year following a chronic kidney disease diagnosis, determine the percentage of patients who experienced their first ischemic stroke within that specific year out of the total initial cohort.
    3. Determine the annual incidence rate of ischemic stroke for patients diagnosed with chronic kidney disease, specifically looking at the percentage of at-risk patients who develop a stroke in year one, year  two, year three, year four, and year five post-diagnosis.
    4. Among patients with a new diagnosis of stage three or higher chronic kidney disease, calculate the proportion who develop an ischemic stroke during each of the five consecutive twelve-month periods following their diagnosis date.
    5. For a cohort of patients with 

## Summary

The disambiguation pipeline resolves implicit ambiguities that are common in epidemiological questions. Key benefits:

1. **Transparency**: Users can review and edit interpretations before SQL generation
2. **Educational value**: Outputs serve as templates for well-formed questions
3. **Systematic**: Applies epidemiological best practices (unique patient counts, temporal precision) without requiring users to specify routine parameters

For the full uncertainty decomposition pipeline (H(I), H(R|I), regime classification), see `clues_demo.ipynb`.

---

## Citation

```bibtex
@misc{ziletti2026disentanglingambiguityinstabilitylarge,
      title={Disentangling Ambiguity from Instability in Large Language Models: A Clinical Text-to-SQL Case Study},
      author={Angelo Ziletti and Leonardo D'Ambrosi},
      year={2026},
      eprint={2602.12015},
      archivePrefix={arXiv},
      primaryClass={cs.CL},
      url={https://arxiv.org/abs/2602.12015},
}
```